# MCP in LangChain

MCP tools get automatically converted to LangChain tools and work with agents like any other tool.

## 🍎 Mac / 🪟 Windows — what's different?

Most of this notebook is the same on both systems. Look out for these markers:

| | 🍎 Mac / Linux | 🪟 Windows |
|---|---|---|
| **Windows fix cell** (after Setup) | Run it anyway — it does nothing on Mac | **Required.** Run it after Setup and *before* Part 1 |
| **Part 1** (time server) | Uses `npx` automatically | Downloads `mcp-time.exe` automatically the first time (needs Node.js / `npm`) |
| **Part 2** (your own server) | Same | Same (works because of the Windows fix cell) |

**Windows: if you restart the kernel**, run the cells again in this order: **Setup → Windows fix → Part 1**.
Restarting wipes the fix, so run the fix cell again after every restart.

**If something fails with `Connection closed`**, open `mcp_server_stderr.log` (in this folder, Windows only) — the MCP server's own error message is there.

**WARNING** there might be a version conflict in your virtual environment. Please run in a separate cell

````python
%pip install -U fastmcp mcp
%pip install -U langchain-mcp-adapters
````

and then restart the kernel **before** continuing!

## Setup


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set! Check your .env file."
assert os.environ.get("OPENAI_ENDPOINT"), "OPENAI_ENDPOINT is not set! Check your P.env file."

from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="gpt-5.4-mini")
llm = ChatOpenAI(
    base_url=os.environ["OPENAI_ENDPOINT"],
    model=os.environ.get("DEPLOYMENT_NAME"),
)

### 🪟 Windows fix (🍎 Mac: just run it, it does nothing)

On Windows, Jupyter can't hand its output stream to the MCP server process, so every stdio MCP connection fails with
`UnsupportedOperation: fileno`. This cell fixes that by writing the server's log/error output to a file, `mcp_server_stderr.log`, instead.

- Run it **after Setup** and **before Part 1**.
- **After a kernel restart, run it again** — a restart removes the fix.

In [ ]:
import sys

if sys.platform == "win32":
    import langchain_mcp_adapters.sessions as _lc_sessions
    from mcp.client.stdio import stdio_client as _orig_stdio_client

    # A real file has a fileno(), unlike Jupyter's sys.stderr
    _mcp_errlog = open("mcp_server_stderr.log", "w", encoding="utf-8")

    def _stdio_client_with_file_errlog(server, errlog=None):
        return _orig_stdio_client(server, errlog=_mcp_errlog)

    _lc_sessions.stdio_client = _stdio_client_with_file_errlog
    print("🪟 Windows fix applied — MCP server logs go to mcp_server_stderr.log")
else:
    print("🍎 Not Windows — no fix needed, carry on.")

## Part 1: Connect to an MCP Server

Connect to the `mcp-time` server using stdio transport.

The cell below picks the right way to start the server for your system:
- 🍎 **Mac / Linux:** runs it with `npx -y @theo.foobar/mcp-time`.
- 🪟 **Windows:** the npm package has a bug that stops it from installing on Windows ("Platform not supported"),
  so the cell downloads the Windows program (`mcp-time.exe`) into a `mcp-time-bin` folder the first time and runs that directly.

Both need Node.js installed (for `npx` / `npm`).

In [ ]:
import sys, shutil, subprocess, tarfile
from pathlib import Path
from langchain_mcp_adapters.client import MultiServerMCPClient

if sys.platform == "win32":
    # 🪟 Windows: download the mcp-time binary once, then run it directly
    bin_dir = Path("mcp-time-bin")
    mcp_time_exe = bin_dir / "package" / "bin" / "mcp-time.exe"
    if not mcp_time_exe.exists():
        print("Downloading mcp-time.exe (first time only)...")
        bin_dir.mkdir(exist_ok=True)
        subprocess.run([shutil.which("npm"), "pack", "mcp-time-windows-x64@0.4.0"],
                       cwd=bin_dir, check=True)
        with tarfile.open(bin_dir / "mcp-time-windows-x64-0.4.0.tgz") as tf:
            tf.extractall(bin_dir, filter="data")
    time_server = {"transport": "stdio", "command": str(mcp_time_exe.resolve()), "args": []}
else:
    # 🍎 Mac / Linux: npx downloads and starts the server
    time_server = {"transport": "stdio", "command": "npx", "args": ["-y", "@theo.foobar/mcp-time"]}

# Connect to the mcp-time server
mcp_client = MultiServerMCPClient({"time": time_server})

# Load tools from the MCP server
mcp_tools = await mcp_client.get_tools()
print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")

### Create an Agent with MCP Tools

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=mcp_tools,
    system_prompt="You are a helpful assistant with access to time-related tools.",
)

### Test the Agent


In [5]:
result = await agent.ainvoke(
    {
        "messages": [
            {"role": "user", "content": "What's the time in San Francisco right now?"}
        ]
    }
)

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What's the time in San Francisco right now?
================================== Ai Message ==================================
Tool Calls:
  current_time (call_DgBZch5n0tlvv2lhcUrAXwdO)
 Call ID: call_DgBZch5n0tlvv2lhcUrAXwdO
  Args:
    timezone: America/Los_Angeles
    format: 2006-01-02 15:04:05 MST
================================= Tool Message =================================
Name: current_time

[{'type': 'text', 'text': '2026-11-13 14:58:02 PST', 'id': 'lc_8b46271a-701e-43ee-943b-aa22e4352fff'}]
================================== Ai Message ==================================

The time in San Francisco right now is **2026-11-13 14:58:02 PST**.


## Part 2: Connect to Your Custom MCP Server

Connect to the MCP server you built in Exercise 1. Make sure it works with the MCP Inspector before continuing.

This part is the same on 🍎 Mac and 🪟 Windows. The first cell only checks that your server starts: you should see the FastMCP banner.
It decodes the output as UTF-8, because the banner's emoji and box characters crash Windows' default decoding.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

uv = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
server_path = str(Path("exercise1-simple-server.py").resolve())
clean_env = {k: v for k, v in os.environ.items() if k not in ("VIRTUAL_ENV", "PYTHONPATH")}
clean_env["PYTHONUTF8"] = "1"   # make the server's Python write UTF-8 (needed on Windows, harmless on Mac)

# Quick check: start the server once and show its startup output (it exits right away because stdin is empty)
r = subprocess.run(
    [uv, "run", "--isolated", "--with", "fastmcp", "python", server_path],
    input="", capture_output=True, timeout=120, env=clean_env,
    text=True, encoding="utf-8", errors="replace",  # utf-8: Windows would otherwise fail on the banner
)
print("STDERR:\n", r.stderr[-2000:])

In [8]:
# Connect to your custom MCP server from Exercise 1
# Make sure you completed Exercise 1 before running this!
my_client = MultiServerMCPClient(
    {
        "my_server": {
            "transport": "stdio",
            "command": uv,
            # "args": ["run", "python", "exercise1 - simple-server.py"],
            "args": ["run", "--isolated", "--with", "fastmcp", "python", server_path],
        }
    }
)

In [9]:
my_tools = await my_client.get_tools()
print(f"Loaded {len(my_tools)} tools: {[t.name for t in my_tools]}")

Loaded 2 tools: ['hello', 'add']


In [10]:
my_agent = create_agent(
    model=llm,
    tools=my_tools,
    system_prompt="You have access to a custom MCP server with greeting and math tools.",
)

In [ ]:
from langchain_core.messages import HumanMessage

result = await my_agent.ainvoke(
    {
        "messages": [HumanMessage(content="Say hello to Alice and then add 5 + 3")]
    }
)
result["messages"][-1].pretty_print()

## Try It Yourself

### Exercise: Combine Multiple MCP Servers

Connect to both the time server and your custom server at the same time. Then ask a question that uses tools from both servers.

**Hint:** `MultiServerMCPClient` accepts multiple server configs in the same dictionary.

**Tip (🍎 + 🪟):** Part 1 stored the time server's config in `time_server`, already set up for your system,
so you can write `"time": time_server` instead of typing the config again.

In [ ]:
# TODO: Create a MultiServerMCPClient with both the "time" server and your custom server
# Hint: Pass both server configs in the same dictionary to MultiServerMCPClient

combined_client = ...

# TODO: Load all tools from both servers and print the full list
all_tools = ...

# TODO: Create an agent with all tools and an appropriate system prompt
combined_agent = ...

# TODO: Invoke the combined agent with a query that requires tools from both servers
# Try: "Say hello to Alice, then tell me what time it is in Tokyo"

## Bonus: Try Public MCP Servers

### HTTP/SSE Servers (Remote - no install needed)

Use `"transport": "sse"` and provide the `"url"` key.

| Server | URL | Description |
|---|---|---|
| Cloudflare Docs | `https://docs.mcp.cloudflare.com/sse` | Search Cloudflare documentation |
| DeepWiki | `https://mcp.deepwiki.com/sse` | Knowledge retrieval |
| Hugging Face | `https://hf.co/mcp` | ML model hub |

```python
MultiServerMCPClient({
    "cloudflare": {
        "transport": "sse",
        "url": "https://docs.mcp.cloudflare.com/sse",
    }
})
```

### npx Servers (Local - no API key needed)

| Server | npx command | Description |
|---|---|---|
| Everything | `npx -y @modelcontextprotocol/server-everything` | Official demo: `echo`, `add`, and more |
| Calculator | `npx -y @wrtnlabs/calculator-mcp@latest` | Math: add, sub, mul, div, sqrt |
| DuckDuckGo | `npx -y duckduckgo-mcp-server` | Web search, no API key |
| Weather | `npx -y @dangahagan/weather-mcp@latest` | Global weather forecasts |

> 🪟 **Windows note:** the npx servers above haven't been tested on Windows. If one fails with `Connection closed`,
> check `mcp_server_stderr.log` for the server's own error message. If `npx` itself isn't found, try
> `"command": "cmd", "args": ["/c", "npx", "-y", "<package>"]`.